# CENG463 PA2

In this programming assignment, you will be dealing with word embeddings and neural networks. You will use Python for this task. You can use libraries such as `pandas`, `nltk`, `numpy` etc. for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results. The assignment consists of 3 questions.

### IMPORTANT NOTE

Do not move or delete the given cells, only add cells inbetween the questions for your answers.

In [ ]:
# UPDATE THIS CELL TO INSTALL NEEDED LIBRARIES.
# MAKE SURE TO ADD EVERYTHING THAT NEEDS TO BE INSTALLED IN THIS CELL!

# we will use pip to install packages - you can add others below
!pip install pandas
!pip install numpy
!pip install nltk
!pip install gensim
!pip install scikit-learn
!pip install tensorflow
!pip install regex
!pip install seaborn

# and import them here - you can add others below
import pandas as pd
import numpy as np
import nltk
import gensim
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import (Model, Sequential)
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Concatenate, Dropout, SimpleRNN
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#for metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# for pa1
import re
import time
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

## Q1 - Word embeddings (50 points)

In this question, you will first train a Word2Vec model, then use it to represent and reason about user reviews.

### Q1.A - training (10 points)
Load the `user_review_train.csv` file shared with you. Using `Word2Vec` module of `gensim.models`, train a **skip-gram** Word2Vec model on the train data.

#### Notes and tips

- Use the given preprocessing function `preprocess_review`.

In [ ]:
# PREPROCESSING FUNCTIONS GIVEN FOR YOU

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')  
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(review):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    sentences = sent_tokenize(review)
    
    lemmatized_review = []
    
    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]
        
        lemmatized_review = lemmatized_review + lemmatized_sentence
    
    return lemmatized_review

In [ ]:
# Q1.A - implementation
# you can add cells below if needed
review_test = pd.read_csv('data/user_review_train.csv')
review_test["preprocessed_review"] = review_test["review"].apply(preprocess_review)
print(review_test["preprocessed_review"].head())

model = gensim.models.Word2Vec(sentences=review_test["preprocessed_review"], 
                               vector_size=100, 
                               window=5, 
                               min_count=5,
                               workers=4, 
                               sg=1,
                               epochs=10)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(model.wv)}")

### Q1.B - word similarity (10 points)

Using the trained model, report the following:

- Similarity between "good" and "bad"
- Similar words to "good"
- Similar words to "bad"
- Similar words to "good" but not similar to "bad"
- Similar words to "good" but not similar to "bad"

and discuss the reported words and scores. Is it possible to identify specific good/bad features of the product that is being reviewed? What other words can be looked up to get more insight?

#### Notes and tips

- Check the [documentation](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.html) of `gensim.models.Word2Vec` to find relevant methods.

In [ ]:
# Q1.B - implementation
# you can add cells below if needed

sim = model.wv.similarity('good', 'bad')  # Example similarity check
print("Similarity between 'good' and 'bad':", sim)

most_similar_words = model.wv.most_similar("good")
print("Words most similar to 'good':", most_similar_words)

most_similar_words = model.wv.most_similar("bad")
print("Words most similar to 'bad':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['good'], negative=['bad'], topn=5)
print("Words similar to 'good' but not 'bad':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['bad'], negative=['good'], topn=5)
print("Words similar to 'bad' but not 'good':", simGoodbutNotBad)

### Seem we cannot get a good results to detect good and bad features. Let's try another pair like worst and best.

In [ ]:
sim = model.wv.similarity('worst', 'best')  # Example similarity check
print("Similarity between 'worst' and 'best':", sim)

In [ ]:
most_similar_words = model.wv.most_similar("worst")
print("Words most similar to 'worst':", most_similar_words)

most_similar_words = model.wv.most_similar("best")
print("Words most similar to 'best':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['worst'], negative=['best'], topn=5)
print("Words similar to 'worst' but not 'best':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['best'], negative=['worst'], topn=5)
print("Words similar to 'bad' but not 'best':", simGoodbutNotBad)

### Q1.B - discussion
Write your discussion here

### Q1.C - representation (15 points)

An important use of word embeddings is representing "documents" (reviews in our case). For this question, before creating the representations, do the following:

- Randomly sample 2 reviews from sentiment label 0, refer to them as sent0_a and sent0_b.
- Randomly sample 2 reviews from sentiment label 1, refer to them as sent1_a and sent1_b.

After the sampling, follow these steps to represent each review:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review, fetch the vector of that token.
- Take the average of the token vectors in the review to represent that review.

Then, calculate and report the cosine similarity of the two vectors representing:
    - sent0_a and sent0_b
    - sent0_a and sent1_a
    - sent1_a and sent1_b

Does this representation work to capture the labels of the reviews? Do you think there is a better way to represent each review instead of taking the average of the word vectors? Discuss your findings with respect to these questions. Repeating the sampling process several times might give you a better insight.

#### Notes and tips

- You can use `numpy` for your calculations.

In [ ]:
# Q1.C - implementation
# you can add cells below if needed
def get_rndm_review(sentiment):
    while True:
        index = np.random.randint(0, len(review_test))
        select_review = review_test["review"][index]
        if(review_test["sentiment"][index] == sentiment):
            print("Selected review index:", index)
            break
    return select_review


np.random.seed(33)
sent1_a = get_rndm_review(1)
sent1_b = get_rndm_review(1)
sent0_a = get_rndm_review(0)
sent0_b = get_rndm_review(0)

print("Positive Review 1:", sent1_a)
print("Positive Review 2:", sent1_b)      
print("Negative Review 1:", sent0_a)
print("Negative Review 2:", sent0_b)

In [ ]:
#preprocess reviews

preprocessed_positive_review_1 = preprocess_review(sent1_a)
preprocessed_positive_review_2 = preprocess_review(sent1_b)
preprocessed_negative_review_1 = preprocess_review(sent0_a)
preprocessed_negative_review_2 = preprocess_review(sent0_b)

print("Preprocessed Positive Review 1:", preprocessed_positive_review_1)
print("Preprocessed Positive Review 2:", preprocessed_positive_review_2)
print("Preprocessed Negative Review 1:", preprocessed_negative_review_1)
print("Preprocessed Negative Review 2:", preprocessed_negative_review_2)


In [ ]:
#For each token in the review, fetch the vector of that token.

def get_review_vector(review):
    review_vector = np.zeros((100,))
    valid_word_count = 0
    for token in review:
        if token in model.wv:  # Check if token exists
            review_vector += model.wv[token]
            valid_word_count += 1
    
    if valid_word_count > 0:  # Avoid division by zero
        review_vector /= valid_word_count
    return review_vector

positive_review_vector_1 = get_review_vector(preprocessed_positive_review_1)
positive_review_vector_2 = get_review_vector(preprocessed_positive_review_2)
negative_review_vector_1 = get_review_vector(preprocessed_negative_review_1)
negative_review_vector_2 = get_review_vector(preprocessed_negative_review_2)

print("Positive Review Vector 1:", positive_review_vector_1)
print("Positive Review Vector 2:", positive_review_vector_2)
print("Negative Review Vector 1:", negative_review_vector_1)
print("Negative Review Vector 2:", negative_review_vector_2)

In [ ]:
# Cosine similarty function

def cos_similarity(vec1, vec2):
    cos_sim = np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))
    return cos_sim

# Calculate cosine similarities
sim_pos_pos = cos_similarity(positive_review_vector_1, positive_review_vector_2)
sim_pos_neg_1 = cos_similarity(positive_review_vector_1, negative_review_vector_1)
sim_neg_neg = cos_similarity(negative_review_vector_1, negative_review_vector_2)


print("Cosine Similarity between two positive reviews:", sim_pos_pos)
print("Cosine Similarity between positive and negative review:", sim_pos_neg_1)
print("Cosine Similarity between two negative reviews:", sim_neg_neg)


In [ ]:
# alternative approach 
def prep(text):
    return preprocess_review(text)

sent0_a_tokens = prep(sent0_a)
sent0_b_tokens = prep(sent0_b)
sent1_a_tokens = prep(sent1_a)
sent1_b_tokens = prep(sent1_b)


train_preprocessed = [" ".join(prep(r)) for r in review_test["review"]]

tfidf = TfidfVectorizer()
tfidf.fit(train_preprocessed)

idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

# 4 — CREATE TF-IDF WEIGHTED WORD2VEC DOCUMENT VECTORS
def document_vector(tokens, model, idf_dict):
    vectors = []
    weights = []

    for tok in tokens:
        if tok in model.wv:
            w = idf_dict.get(tok, 0.0)  # TF-IDF weight
            vectors.append(model.wv[tok] * w)
            weights.append(w)

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.sum(vectors, axis=0) / (np.sum(weights) + 1e-9)

vec0a = document_vector(sent0_a_tokens, model, idf_dict)
vec0b = document_vector(sent0_b_tokens, model, idf_dict)
vec1a = document_vector(sent1_a_tokens, model, idf_dict)
vec1b = document_vector(sent1_b_tokens, model, idf_dict)

# 5 — COSINE SIMILARITIES
sim_00 = cosine_similarity([vec0a], [vec0b])[0][0]
sim_01 = cosine_similarity([vec0a], [vec1a])[0][0]
sim_11 = cosine_similarity([vec1a], [vec1b])[0][0]

print("sim(sent0_a, sent0_b):", sim_00)
print("sim(sent0_a, sent1_a):", sim_01)
print("sim(sent1_a, sent1_b):", sim_11)

### Q1.C - discussion
Write your discussion here

### Q1.D - training and comparing classifiers (15 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train a binary classification model with Word2Vec representations, and compare its performance with a binary classifier using Bag-of-Words representation.

As the Bag-of-Words classifier, you can either choose the best performing classifier you have implemented in Question 3 of Programming Assignment 1, or you can follow these steps:

- Preprocess the review with the given `preprocess_review` function.
- Order all unique tokens by frequency, take the most frequent 100.
- Use these 100 words as the corpus for Bag-of-Words representation.

For the Word2Vec model, represent the reviews by following these steps:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review that is also in the most frequent 100 tokens, fetch the vector of that token.
- Take the average of the token vectors selected to represent that review.

After training both classifiers on `user_review_train.csv`, test them with `user_review_test.csv` and report the performance of your models with four metrics: accuracy, precision, recall and F1-score. Compare the performance of both models and discuss in detail.

#### Notes and tips

- You can use `CountVectorizer` from `scikit-learn` or any other library available for Bag-of-Words representation.
- You should select a classification method from the following set of classifiers: `[Naive Bayes, Support Vector Machine, Logistic Regression, Random Forest]`. You can use `scikit-learn`, `nltk`, or any other library for the classifier implementations. 
- You should **not** use the test set `user_reviews_test.csv` during your training process. You should use `user_reviews_train.csv` only.
- You may add a validation step in your training process. To do this, you can further split the `user_reviews_train.csv` data and apply k-fold cross validation.

### TAKEN FROM MY PA1 PIECE OF CODE

In [ ]:
# Preprocessing helper functions


train_data = pd.read_csv('data/user_review_train.csv')
test_data = pd.read_csv('data/user_review_test.csv')

X_train = train_data['review'].values
y_train = train_data['sentiment'].values

X_test = test_data['review'].values
y_test = test_data['sentiment'].values



In [ ]:
tokenized_train = train_data['review'].apply(preprocess_review).tolist()
tokenized_test = test_data['review'].apply(preprocess_review).tolist()

all_tokens = [t for r in tokenized_train for t in r]
top_500_tokens = [token for token, freq in Counter(all_tokens).most_common(500)]

train_strings = [' '.join(r) for r in tokenized_train]
test_strings = [' '.join(r) for r in tokenized_test]

In [ ]:
def get_bow_features(processed_reviews, vocabulary):
    
    # Convert processed reviews into Bag-of-Words features using a fixed vocabulary.
    vectorizer = CountVectorizer(vocabulary=vocabulary)
    X = vectorizer.fit_transform(processed_reviews)
    return X

x_train_bow = get_bow_features(train_strings, top_500_tokens)
x_test_bow = get_bow_features(test_strings, top_500_tokens)

In [ ]:
w2v_model = gensim.models.Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=1,   # keep all words for now
    sg=1,
    epochs=10
)

In [ ]:
def review_to_w2v_avg(tokens, model, top_tokens, vector_size=100):
    vecs = [model.wv[t] for t in tokens if t in top_tokens and t in model.wv]
    if len(vecs) == 0:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0) # take average of word vectors

X_train_w2v = np.array([review_to_w2v_avg(r, w2v_model, top_500_tokens) for r in tokenized_train])
X_test_w2v = np.array([review_to_w2v_avg(r, w2v_model, top_500_tokens) for r in tokenized_test])

In [ ]:
# I prefer to use RandomForest since I have got higher performence in PA1.

# --- BoW classifier ---
clf_bow = RandomForestClassifier(n_estimators=100, random_state=42)
clf_bow.fit(x_train_bow, y_train)
y_pred_bow = clf_bow.predict(x_test_bow)

# --- Word2Vec classifier ---
clf_w2v = RandomForestClassifier(n_estimators=100, random_state=42)
clf_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_test_w2v)

In [ ]:
def evaluate(y_true, y_pred, name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name} Performance:")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}\n")

evaluate(y_test, y_pred_bow, "Bag-of-Words Classifier")
evaluate(y_test, y_pred_w2v, "Word2Vec Classifier")

### Q1.D - discussion

**Performance Comparison:**



## Q2 - Neural Networks for Binary Classification (50 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train two neural network models for the binary classification of user reviews and compare their performances. You are expected to train RNN (part A - 20 points) and TextCNN (part B - 20 points) models, and report the following: 

- Confusion matrix of both models
- Time it took to train both models
- Accuracy, precision, recall, and F1-score of both models
- Other metrics you think are important

Finally (part C), you should discuss the performance of the models according to your reported results. Try to analyse the models in terms of pros and cons of using each one.

#### Notes and tips

- For the embedding layers of the models, you are free to use word embedding methods or leave them randomly initialised. Similarly, you can use word-based or character-based embeddings. However, make sure to explain your decisions.
- You are expected to use `tensorflow` for your implementations, but you can use other libraries if you already have a working setup.

In [ ]:
# Q2.A - implementation of RNN
# you can add cells below if needed
train_data = pd.read_csv('data/user_review_train.csv')
test_data = pd.read_csv('data/user_review_test.csv')

X_train = train_data['review'].values
y_train = train_data['sentiment'].values

X_test = test_data['review'].values
y_test = test_data['sentiment'].values


In [ ]:


# ensure we've read the data correctly (using head() to avoid too much output)
print(train_data.head())
print(test_data.head())

def clean_text(t):
    t = t.lower()
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Preprocess the text data. 
# It is different then previous state of art NLP preprocessing such as stop word removal and lemmatization. 
# Because RNNs can learn from the full sequences, we only lowercase and remove extra spaces here.
X_train = [clean_text(x) for x in X_train]
X_test = [clean_text(x) for x in X_test]

In [ ]:
# Tokenize and pad sequences

# Fit tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Dynamic vocab size
vocab_size = len(tokenizer.word_index) + 1
print("VOCAB SIZE =", vocab_size)

# Compute dynamic max_len (95th percentile) to eliminate outliers in training data
sequences = tokenizer.texts_to_sequences(X_train)
lengths = [len(s) for s in sequences]
max_len = int(np.percentile(lengths, 95))
print("MAX_LEN =", max_len)

# choose embedding dimension as 64 for this example. No special reason for this value other than being a power of 2 and not too large.
embedding_dim = 64

#those paramters will be used for RNN and TextCNN models
print("EMBEDDING DIM =", embedding_dim)

# Important Note: I prefer to calculate vocab_size and max_len dynamically from the training data
# rather than using fixed values although using hard coded values are used in simple example on internet. 
# This allows the model to adapt better to the actual data distribution I hope :)

# Pad using dynamic max_len
X_train_pad = pad_sequences(sequences, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=max_len,
    padding="post"
)

In [ ]:
# Build RNN model now 


# define the RNN model. Keep it simple with one SimpleRNN layer. Hidden layer size is chosen as 64.
# since we will use this model sentiment analysis, final layer has 1 neuron with sigmoid activation. It is (binary classification).
rnn_model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

# Build the model by specifying input shape. 
# I need to put this here to show the model summary later. If not called here, model.summary() will give eöpty.
rnn_model.build(input_shape=(None, max_len))

# classic compile step for binary classification
rnn_model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

rnn_model.summary()


In [ ]:
# train the rnn model

start_time = time.time()
history_rnn = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=5,
    batch_size=32
)
rnn_training_time = time.time() - start_time
print(f"\nRNN Training Time: {rnn_training_time:.2f} seconds")


In [ ]:
# RNN Model Evaluation

# Evaluate on test set
loss_rnn, acc_rnn = rnn_model.evaluate(X_test_pad, y_test, verbose=0)
print("RNN Test Accuracy:", acc_rnn)
print("RNN Test Loss:", loss_rnn)

# Get predictions
y_pred_proba_rnn = rnn_model.predict(X_test_pad, verbose=0)
y_pred_rnn = (y_pred_proba_rnn > 0.5).astype(int).flatten()

# Calculate metrics
acc = accuracy_score(y_test, y_pred_rnn)
prec = precision_score(y_test, y_pred_rnn)
rec = recall_score(y_test, y_pred_rnn)
f1 = f1_score(y_test, y_pred_rnn)

print("\n=== RNN Model Performance ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Training Time: {rnn_training_time:.2f} seconds")

# Confusion Matrix
cm_rnn = confusion_matrix(y_test, y_pred_rnn)
print("\nConfusion Matrix:")
print(cm_rnn)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('RNN Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rnn, target_names=['Negative', 'Positive']))


In [ ]:
# Q2.B - implementation of TextCNN
# you can add cells below if needed


input_layer = Input(shape=(max_len,))

embedding = Embedding(vocab_size, embedding_dim, input_length=max_len)(input_layer)

# parallel convolution layers with different kernel sizes
conv_3 = Conv1D(filters=64, kernel_size=3, activation='relu')(embedding)
conv_4 = Conv1D(filters=64, kernel_size=4, activation='relu')(embedding)
conv_5 = Conv1D(filters=64, kernel_size=5, activation='relu')(embedding)

# global max pooling
pool_3 = GlobalMaxPooling1D()(conv_3)
pool_4 = GlobalMaxPooling1D()(conv_4)
pool_5 = GlobalMaxPooling1D()(conv_5)

# concatenate
concat = Concatenate()([pool_3, pool_4, pool_5])

dropout = Dropout(0.5)(concat)

output = Dense(1, activation='sigmoid')(dropout)

textcnn_model = Model(inputs=input_layer, outputs=output)

textcnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

textcnn_model.summary()


In [ ]:
# Train TextCNN model

start_time = time.time()
history_textcnn = textcnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=5,
    batch_size=32
)
textcnn_training_time = time.time() - start_time
print(f"\nTextCNN Training Time: {textcnn_training_time:.2f} seconds")


In [ ]:
# TextCNN Model Evaluation

# Evaluate on test set
loss_textcnn, acc_textcnn = textcnn_model.evaluate(X_test_pad, y_test, verbose=0)
print("TextCNN Test Accuracy:", acc_textcnn)
print("TextCNN Test Loss:", loss_textcnn)

# Get predictions
y_pred_proba_textcnn = textcnn_model.predict(X_test_pad, verbose=0)
y_pred_textcnn = (y_pred_proba_textcnn > 0.5).astype(int).flatten()

# Calculate metrics
acc = accuracy_score(y_test, y_pred_textcnn)
prec = precision_score(y_test, y_pred_textcnn)
rec = recall_score(y_test, y_pred_textcnn)
f1 = f1_score(y_test, y_pred_textcnn)

print("\n=== TextCNN Model Performance ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Training Time: {textcnn_training_time:.2f} seconds")

# Confusion Matrix
cm_textcnn = confusion_matrix(y_test, y_pred_textcnn)
print("\nConfusion Matrix:")
print(cm_textcnn)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_textcnn, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('TextCNN Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_textcnn, target_names=['Negative', 'Positive']))


In [ ]:
# Comparison Summary

print("\n" + "="*60)
print("COMPARISON SUMMARY: RNN vs TextCNN")
print("="*60)

# Create comparison dataframe
comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Training Time (s)'],
    'RNN': [
        accuracy_score(y_test, y_pred_rnn),
        precision_score(y_test, y_pred_rnn),
        recall_score(y_test, y_pred_rnn),
        f1_score(y_test, y_pred_rnn),
        rnn_training_time
    ],
    'TextCNN': [
        accuracy_score(y_test, y_pred_textcnn),
        precision_score(y_test, y_pred_textcnn),
        recall_score(y_test, y_pred_textcnn),
        f1_score(y_test, y_pred_textcnn),
        textcnn_training_time
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Difference'] = comparison_df['TextCNN'] - comparison_df['RNN']
print(comparison_df.to_string(index=False))

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
rnn_scores = [accuracy_score(y_test, y_pred_rnn), 
              precision_score(y_test, y_pred_rnn),
              recall_score(y_test, y_pred_rnn),
              f1_score(y_test, y_pred_rnn)]
textcnn_scores = [accuracy_score(y_test, y_pred_textcnn),
                  precision_score(y_test, y_pred_textcnn),
                  recall_score(y_test, y_pred_textcnn),
                  f1_score(y_test, y_pred_textcnn)]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, rnn_scores, width, label='RNN', color='skyblue')
axes[0].bar(x + width/2, textcnn_scores, width, label='TextCNN', color='lightgreen')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=45)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Training time comparison
axes[1].bar(['RNN', 'TextCNN'], [rnn_training_time, textcnn_training_time], 
            color=['skyblue', 'lightgreen'])
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


### Q2.C - discussion

Write your discussion here.